In [1]:
# ===== P3 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P3 CELL 2: Load data, settings, and check the protocol is unchanged =====
import os, json, time, hashlib
import numpy as np
import pandas as pd

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
CACHE_DIR = os.path.join(BASE, "02_data_processed", "cache")
FEAT_DIR = os.path.join(BASE, "02_data_processed", "features")
RES_DIR = os.path.join(BASE, "05_results")
os.makedirs(FEAT_DIR, exist_ok=True)

with open(os.path.join(BASE, "config.json")) as fh:
    CONFIG = json.load(fh)
proto = CONFIG["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    assert hashlib.sha256(fh.read()).hexdigest() == proto["sha256"], "Protocol file changed - STOP"
PLAN = proto["plan"]
SEED = PLAN["random_seed"]
print("Protocol", proto["version"], "verified unchanged (frozen at", proto["frozen_at"] + ")")

data = {name: pd.read_pickle(os.path.join(CACHE_DIR, f"{name}.pkl")) for name in ["train", "val", "test"]}
for name, df in data.items():
    print(f"{name:5s}: {len(df):,} rows")

Protocol v1 verified unchanged (frozen at 2026-10-19T05:54:18)
train: 763,144 rows
val  : 188,967 rows
test : 188,967 rows


In [3]:
# ===== P3 CELL 3: Learn the preprocessing settings from TRAINING data only =====
tr = data["train"]

proc_counts = tr["processName"].astype(str).value_counts()
PREP = {
    "process_vocab": sorted(proc_counts.index.tolist()),                       # names seen in training
    "event_vocab": sorted(tr["eventName"].astype(str).unique().tolist()),
    "process_logfreq": {k: float(np.log1p(v)) for k, v in proc_counts.items()},  # unseen names -> 0
    "host_mount_namespace": 4026531840,
    "os_process_ids": [0, 1, 2],
    "user_threshold": 1000,
    "fitted_on": "train split only",
}
with open(os.path.join(FEAT_DIR, "preprocessing_settings.json"), "w") as fh:
    json.dump(PREP, fh, indent=2)

print(f"processName vocabulary (train): {len(PREP['process_vocab'])} names")
print(f"eventName vocabulary (train):   {len(PREP['event_vocab'])} names")
print("Most common training process names (log frequency):")
for k in proc_counts.index[:5]:
    print(f"   {k:20s} count={proc_counts[k]:>8,}  log1p={PREP['process_logfreq'][k]:.2f}")

processName vocabulary (train): 36 names
eventName vocabulary (train):   32 names
Most common training process names (log frequency):
   ps                   count= 406,313  log1p=12.91
   systemd-udevd        count= 189,292  log1p=12.15
   sshd                 count=  91,762  log1p=11.43
   systemd-journal      count=  20,421  log1p=9.92
   systemd              count=  17,310  log1p=9.76


In [4]:
# ===== P3 CELL 4: Build the four feature sets for every split and save them =====
def stack_length(s):
    s = s.astype(str).str.strip()
    return np.where(s == "[]", 0, s.str.count(",") + 1).astype(np.float32)

def one_hot(values, vocab, prefix):
    known = pd.Categorical(values.astype(str), categories=vocab)
    mat = np.zeros((len(values), len(vocab) + 1), dtype=np.float32)
    codes = known.codes                          # -1 means "not in training vocabulary"
    mat[np.arange(len(values)), np.where(codes >= 0, codes, len(vocab))] = 1.0
    return mat, [f"{prefix}={v}" for v in vocab] + [f"{prefix}=UNKNOWN"]

def build_features(df):
    base = pd.DataFrame({
        "is_os_process": df["processId"].isin(PREP["os_process_ids"]).astype(np.float32),
        "is_os_parent": df["parentProcessId"].isin(PREP["os_process_ids"]).astype(np.float32),
        "user_ge_1000": (df["userId"] >= PREP["user_threshold"]).astype(np.float32),
        "host_mount_ns": (df["mountNamespace"] == PREP["host_mount_namespace"]).astype(np.float32),
        "return_sign": np.sign(df["returnValue"]).astype(np.float32),
        "argsNum": df["argsNum"].astype(np.float32),
    })
    proc_lf = df["processName"].astype(str).map(PREP["process_logfreq"]).fillna(0.0).astype(np.float32)
    stack_len = stack_length(df["stackAddresses"])
    ev_mat, ev_names = one_hot(df["eventName"], PREP["event_vocab"], "event")
    pr_mat, pr_names = one_hot(df["processName"], PREP["process_vocab"], "process")

    main_cols = list(base.columns) + ["process_logfreq", "stack_len"]
    F_main = np.hstack([base.values, proc_lf.values[:, None], stack_len[:, None], ev_mat, pr_mat])
    names_main = main_cols + ev_names + pr_names

    keep = [i for i, n in enumerate(names_main) if n != "user_ge_1000"]
    F_no_user = F_main[:, keep]
    names_no_user = [names_main[i] for i in keep]

    F_paper = np.column_stack([base["is_os_process"], base["is_os_parent"], base["user_ge_1000"],
                               base["host_mount_ns"], df["eventId"].astype(np.float32),
                               base["argsNum"], base["return_sign"]]).astype(np.float32)
    names_paper = ["is_os_process", "is_os_parent", "user_ge_1000", "host_mount_ns",
                   "eventId", "argsNum", "return_sign"]

    proc_code = pd.Categorical(df["processName"].astype(str), categories=PREP["process_vocab"]).codes
    ev_code = pd.Categorical(df["eventName"].astype(str), categories=PREP["event_vocab"]).codes
    F_raw = np.column_stack([df["timestamp"], df["processId"], df["threadId"], df["parentProcessId"],
                             df["userId"], df["argsNum"], df["returnValue"],
                             ev_code, proc_code]).astype(np.float32)   # unseen names -> -1
    names_raw = ["timestamp", "processId", "threadId", "parentProcessId", "userId",
                 "argsNum", "returnValue", "eventName_code", "processName_code"]

    return {"F_main": (F_main, names_main), "F_no_user": (F_no_user, names_no_user),
            "F_paper": (F_paper, names_paper), "F_raw": (F_raw, names_raw)}

feature_names = {}
for split, df in data.items():
    t0 = time.time()
    sets = build_features(df)
    for fs, (mat, names) in sets.items():
        np.save(os.path.join(FEAT_DIR, f"{fs}_{split}.npy"), mat)
        feature_names[fs] = names
    # labels and identifiers are stored SEPARATELY from features
    meta = df[["row_id", "hostName", "processId", "processName", "eventName",
               "timestamp", "userId", "sus", "evil"]].copy()
    meta["hostName"] = meta["hostName"].astype(str)
    meta["processName"] = meta["processName"].astype(str)
    meta["eventName"] = meta["eventName"].astype(str)
    meta["process_key"] = meta["hostName"] + "|" + meta["processId"].astype(str)
    meta.to_pickle(os.path.join(FEAT_DIR, f"meta_{split}.pkl"))
    print(f"{split:5s}: " + ", ".join(f"{fs} {m.shape[1]} cols" for fs, (m, _) in sets.items())
          + f"  ({time.time() - t0:.1f}s)")

with open(os.path.join(FEAT_DIR, "feature_names.json"), "w") as fh:
    json.dump(feature_names, fh, indent=2)

# Seeded 10% of training rows for autoencoder early stopping (protocol section 3)
rng = np.random.default_rng(SEED)
n_train = len(data["train"])
es_idx = np.sort(rng.choice(n_train, size=n_train // 10, replace=False))
np.save(os.path.join(FEAT_DIR, "ae_early_stopping_idx.npy"), es_idx)
print(f"\nAutoencoder early-stopping rows: {len(es_idx):,} of {n_train:,} training rows")

train: F_main 78 cols, F_no_user 77 cols, F_paper 7 cols, F_raw 9 cols  (8.8s)
val  : F_main 78 cols, F_no_user 77 cols, F_paper 7 cols, F_raw 9 cols  (2.4s)
test : F_main 78 cols, F_no_user 77 cols, F_paper 7 cols, F_raw 9 cols  (2.7s)

Autoencoder early-stopping rows: 76,314 of 763,144 training rows


In [5]:
# ===== P3 CELL 5: Sanity and leakage checks, then record decision D3 =====
import datetime
checks = []
for fs, names in feature_names.items():
    assert not any(n in ("sus", "evil") for n in names), f"label column inside {fs}!"
    for split in ["train", "val", "test"]:
        X = np.load(os.path.join(FEAT_DIR, f"{fs}_{split}.npy"), mmap_mode="r")
        assert X.shape[0] == len(data[split]), "row count mismatch"
        assert np.isfinite(X).all(), f"NaN or infinity in {fs}_{split}"
    Xtr = np.load(os.path.join(FEAT_DIR, f"{fs}_train.npy"))
    const = [names[i] for i in np.where(Xtr.std(axis=0) == 0)[0]]
    checks.append({"feature_set": fs, "columns": len(names),
                   "constant_in_train": len(const), "constant_examples": ", ".join(const[:3])})
check_df = pd.DataFrame(checks)
check_df.to_csv(os.path.join(RES_DIR, "P3_feature_set_summary.csv"), index=False)
print(check_df.to_string(index=False))

unk = feature_names["F_main"].index("process=UNKNOWN")
print("\nRows using an UNKNOWN process name (column process=UNKNOWN):")
for split in ["train", "val", "test"]:
    X = np.load(os.path.join(FEAT_DIR, f"F_main_{split}.npy"), mmap_mode="r")
    print(f"   {split:5s}: {100 * X[:, unk].mean():.2f}%")
print("\nAll leakage and integrity checks passed: no labels in features, no missing values,")
print("vocabularies and frequencies fitted on training data only.")

with open(os.path.join(BASE, "config.json")) as fh:
    config = json.load(fh)
if not any(d.get("id") == "D3" for d in config["decisions"]):
    config["decisions"].append({
        "id": "D3",
        "decision": "Preprocessing details for the pre-registered feature sets.",
        "details": [
            "hostName, timestamp, threadId and raw IDs excluded from F_main (host-specific); used only in F_raw.",
            "args excluded (free-text argument lists; not used by the BETH baselines).",
            "eventName one-hot in F_main; eventId used only in F_paper (one-to-one with eventName).",
            "stackAddresses reduced to its length (68% of training rows are empty lists).",
            "mountNamespace kept as a flag (equals 4026531840), not dropped: it has 6 values.",
            "userId kept only as the flag userId >= 1000.",
            "Unseen process/event names map to UNKNOWN; process_logfreq = 0 for unseen names.",
            "All vocabularies and frequencies fitted on the training split only.",
        ],
        "recorded_at": datetime.datetime.now().isoformat(timespec="seconds"),
    })
    with open(os.path.join(BASE, "config.json"), "w") as fh:
        json.dump(config, fh, indent=2)
    print("\nDecision D3 recorded in config.json")
else:
    print("\nDecision D3 already recorded")

feature_set  columns  constant_in_train              constant_examples
     F_main       78                  2 event=UNKNOWN, process=UNKNOWN
  F_no_user       77                  2 event=UNKNOWN, process=UNKNOWN
    F_paper        7                  0                               
      F_raw        9                  0                               

Rows using an UNKNOWN process name (column process=UNKNOWN):
   train: 0.00%
   val  : 0.06%
   test : 89.25%

All leakage and integrity checks passed: no labels in features, no missing values,
vocabularies and frequencies fitted on training data only.

Decision D3 recorded in config.json
